<a href="https://colab.research.google.com/github/yaseen20051/crash-detection-cnn/blob/Branch_N/ahmed_crash_detection_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
import cv2
import os
import random

# **Read Data**

In [24]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ckay16/accident-detection-from-cctv-footage")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'accident-detection-from-cctv-footage' dataset.
Path to dataset files: /kaggle/input/accident-detection-from-cctv-footage


# **Directories**

In [25]:
BASE_DIR = os.path.join(path, 'data')

train_dir = os.path.join(BASE_DIR, 'train')
val_dir = os.path.join(BASE_DIR, 'val')
test_dir=os.path.join(BASE_DIR, 'test')

print("train",train_dir)
print("val",val_dir)
print("test",test_dir)



train /kaggle/input/accident-detection-from-cctv-footage/data/train
val /kaggle/input/accident-detection-from-cctv-footage/data/val
test /kaggle/input/accident-detection-from-cctv-footage/data/test


In [26]:
classes = ["Accident", "Non Accident"]
for c in classes:
    train_path = os.path.join(train_dir, c)
    val_path = os.path.join(val_dir, c)
    test_path = os.path.join(test_dir, c)

# **Functions**

In [27]:
IMAGE_SIZE = (128,128)  # 64 * 64 , 224*224 # CONSTANT
BATCH_SIZE = 32

def load_images_and_labels(directory, label):
    images = []
    labels = []
    # List only files that are likely images to avoid errors with directories or hidden files
    image_files = [f for f in os.listdir(directory) if f.lower().endswith(('.png', '.jpg', '.jpeg'))] # mango1.jpg,mango2.jpg,mango3.jpg
    for filename in image_files:
        filepath = os.path.join(directory, filename)
        image = cv2.imread(filepath) # Use OpenCV to read imag
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, IMAGE_SIZE) # Resize image
        images.append(image)
        labels.append(label)
    return images, labels

# **Load Accident**

In [28]:
train_accident, y_train_accident = load_images_and_labels(
    os.path.join(train_dir, "Accident"),
    1
)

val_accident, y_val_accident = load_images_and_labels(
    os.path.join(val_dir, "Accident"),
    1
)

test_accident, y_test_accident = load_images_and_labels(
    os.path.join(test_dir, "Accident"),
    1
)

print("Train Accident:", len(train_accident))
print("Validation Accident:", len(val_accident))
print("Test Accident:", len(test_accident))

Train Accident: 369
Validation Accident: 46
Test Accident: 47


# **Load Non Accident**

In [29]:
train_non_accident, y_train_non_accident = load_images_and_labels(
    os.path.join(train_dir, "Non Accident"),
    0
)

val_non_accident, y_val_non_accident = load_images_and_labels(
    os.path.join(val_dir, "Non Accident"),
    0
)

test_non_accident, y_test_non_accident = load_images_and_labels(
    os.path.join(test_dir, "Non Accident"),
    0
)

print("Train Non Accident:", len(train_non_accident))
print("Validation Non Accident:", len(val_non_accident))
print("Test Non Accident:", len(test_non_accident))

Train Non Accident: 422
Validation Non Accident: 52
Test Non Accident: 53


# **Combine and shuffle training data**

In [30]:
x_train = np.array(train_accident + train_non_accident)
y_train = np.array(y_train_accident + y_train_non_accident)

print("x_train shape:",x_train.shape)
print("y_train shape:",y_train.shape)

x_train shape: (791, 128, 128, 3)
y_train shape: (791,)


# **Combine and shuffle validation data**

In [31]:
x_val = np.array(val_accident + val_non_accident)
y_val = np.array(y_val_accident + y_val_non_accident)

print("x_val shape:",x_val.shape)
print("y_val shape:",y_val.shape)

x_val shape: (98, 128, 128, 3)
y_val shape: (98,)


# **Combine and shuffle test data**

In [32]:
x_test = np.array(test_accident + test_non_accident)
y_test = np.array(y_test_accident + y_test_non_accident)

print("x_test shape:",x_test.shape)
print("y_test shape:",y_test.shape)

x_test shape: (100, 128, 128, 3)
y_test shape: (100,)
